In [2]:
import warnings
warnings.filterwarnings("ignore")
import logging
logging.getLogger("langchain").setLevel(logging.ERROR)

#Requirements

In [3]:
# Install dependencies
!pip install --upgrade langchain faiss-cpu sentence-transformers google-colab
!pip install langchain-core
!pip install pypdf
!pip install langchain-text-splitters
!pip install langchain_huggingface
!pip install langchain_community
!pip install langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.1/114.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.7/588.7 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 128.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
  Attempting uninstall: langgraph-checkpoint
    Found existing installation: langgraph-checkpoint 4.0.2
    Uninstalling langgraph-checkpoint-4.0.2:
      Successfully uninstalled

In [4]:
GROQ_API_KEY="api_key"

#Document Loading:
We load the document either using simple text, or urls or using pdfs.

In [5]:
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader

# doc_text = """
# Elon Musk is a technology entrepreneur and engineer known for founding SpaceX and Tesla.
# He was born on June 28, 1971, in Pretoria, South Africa.
# His major achievements include advancing space exploration and electric vehicles.
# Musk is also involved with Neuralink and The Boring Company.
# This document provides a brief overview of Musk's background and accomplishments.
# """

urls = [
    "https://arxiv.org/abs/1706.03762",
    "https://arxiv.org/abs/2005.14165"
]
loader = WebBaseLoader(urls)

document = loader.load()

#For pdf following method


In [6]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("https://arxiv.org/pdf/1706.03762.pdf")
documents = loader.load()
print("Document Loading done")

Document Loading done


#Text Splitter:
We split the text from documents into chunks so that we can create embeddings and store them in a vector database

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs_split = splitter.split_documents(documents)

#Embeddings

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    show_progress=False  # ✅ disables the widget
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
from langchain_community.vectorstores import FAISS

We use FAISS for creating vector store with similarity with the documents

In [10]:
vectorstore = FAISS.from_documents(docs_split, embeddings)

In [18]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.0,
    max_tokens=1024,
    api_key=GROQ_API_KEY,
)

In [12]:
from langchain_classic.chains import RetrievalQA

#Retriever

In [19]:
retriever = vectorstore.as_retriever()

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,  # to get source documents with answers
    chain_type="stuff",            # concatenates retrieved docs into prompt
)

#Generating Output

In [20]:
query = "What is the attention mechanism in transformers?"

result = qa_chain.invoke({"query": query})

print("Answer:", result['result'])
# print("\nSource Documents:")
# for doc in result['source_documents']:
#     print(f"- {doc.page_content}")

Answer: The attention mechanism in transformers is a self-attention mechanism that allows the model to attend to different parts of the input sequence simultaneously and weigh their importance. It is a key component of the transformer architecture and is used in both the encoder and decoder stacks.

In the context of the provided text, it is mentioned that the attention mechanism is used to follow long-distance dependencies in the input sequence. For example, in Figure 3, the attention mechanism is shown to attend to a distant dependency of the verb "making", completing the phrase "making...more difficult".

The text also mentions that the transformer uses a multi-head attention mechanism, which allows the model to jointly attend to information from different representation subspaces at different positions. Additionally, it is mentioned that the transformer uses scaled dot-product attention, which is a type of attention mechanism that is used to compute the attention weights.

Overall,

In [23]:
!git remote add origin https://github.com/AyaanK123/RAG_Langchain.git
!git branch -M main
!git push -u origin main

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
